# Day 4 — Complexity Audit

Task for today: go back to **every** problem I have solved so far and write down its **time complexity** (how the number of steps grows with input size) and **space complexity** (how much *extra* memory it needs).

Method: don't guess. **Count the loop iterations.** The loop is where all the work happens.

All functions below are copied exactly from my `Day-01/DSA/main.py` and `Day-04/DSA/main.py`.

## Part 1 — Day 1 maths problems

One idea unlocks almost all of them: **a loop that strips one digit per iteration runs once per digit, not once per value.** A number `n` has about `log10(n) + 1` digits. So these loops are `O(log n)` — even a 15-digit number loops just 15 times.

In [ ]:
def Sum_of_first_N(self,n):
    sum = 0
    for i in range(1,n+1):
        sum += i
    return sum

### Why `Sum_of_first_N` is O(n) time, O(1) space

Count the iterations: `range(1, n+1)` produces `1, 2, 3, ..., n` — exactly **n iterations**. Each iteration does one addition and one assignment (a constant amount of work). Total steps ≈ `2n` → drop the constant → **O(n)**.

- `n = 10` → 10 loop runs. `n = 1,000,000` → 1,000,000 loop runs. Work grows in a straight line with n.
- Space: only `sum` and `i` exist, no matter how big n gets → **O(1)** auxiliary space.

Interview bonus: the formula `n*(n+1)//2` gives the same answer in **O(1)** time — one multiplication, one division. Same chai, zero queue.

In [ ]:
def reverse_a_number(self,number):
    reverse = 0
    while number > 0:
        digit = number % 10
        reverse = reverse *10 + digit
        number //= 10
    return reverse

### Why `reverse_a_number` is O(log n) time, O(1) space

Count the iterations: every pass does `number //= 10`, which **removes one digit**. The loop stops when no digits are left. So:

- `453` (3 digits) → 3 iterations.
- `1,082,945` (7 digits) → 7 iterations.
- A d-digit number → d iterations, and `d ≈ log10(number) + 1`.

Iterations = number of digits = **O(log₁₀ n)**, written **O(log n)** (log bases only differ by a constant, and constants get dropped).

Space: just `reverse` and `digit` → **O(1)**.

In [ ]:
def count_digits(self,num):
    count = 0
    while num > 0:
        num % 10
        count += 1
        num //=10
    return count

### Why `count_digits` is O(log n) time, O(1) space

Exactly the same counting as before: `num //= 10` chops one digit per iteration, so the loop runs **once per digit** → **O(log₁₀ n)** iterations.

This one is neat because the answer *is* the iteration count: `count_digits(1082945)` returns 7 because the loop ran 7 times.

(Small observation while auditing: the line `num % 10` computes a remainder and throws it away — it isn't needed here. It doesn't change the Big-O, just a wasted constant per iteration.)

Space: `count` only → **O(1)**.

In [ ]:
def palindrome_number(self,num):
    number = num
    reverse = 0
    while number > 0:
        digit = number % 10
        reverse = reverse *10 + digit
        number //= 10
    return num == reverse

### Why `palindrome_number` is O(log n) time, O(1) space

It is `reverse_a_number` plus **one** extra comparison at the end.

- Loop: one iteration per digit → O(log n).
- Final `num == reverse`: 1 step → O(1).
- Total: O(log n + 1) = **O(log n)** (drop the smaller term).

Space: `number`, `reverse`, `digit` — three variables regardless of input size → **O(1)**.

In [ ]:
def armstrong_number(self,num):
    temp = num
    count = 0
    while temp > 0:
        temp % 10
        count += 1
        temp //=10


    temp = num
    sum = 0
    while temp>0:
        digit = temp % 10
        sum += digit ** count 
        temp //= 10

    return sum == num

### Why `armstrong_number` is O(log n) time, O(1) space

Two digit-loops, but they run **one after another**, not nested:

1. First loop counts the digits → d iterations (d = number of digits).
2. Second loop computes `digit ** count` for each digit → d iterations again.

Sequential loops **add**: O(log n + log n) = O(2 log n) → drop the constant → **O(log n)**.

Compare with *nested* loops, which **multiply** — that would have been O(log n × log n). Order of loops matters: side by side = add, one inside the other = multiply.

Space: `temp`, `count`, `sum`, `digit` → **O(1)**.

## Part 2 — Patterns 1 to 18

All 18 patterns have the same skeleton: **outer loop over rows, inner loop(s) over columns**. So instead of auditing 18 times, I audit the *shapes* once.

The honest operation count for a pattern = **how many characters get printed** (each `print(..., end="")` is one operation). Three shapes cover everything:

1. **Square** (pattern01): n rows × n stars = `n²` prints.
2. **Triangle** (patterns 02–06, 11, 13–16, 18): row i prints i characters → `1 + 2 + ... + n = n(n+1)/2` prints.
3. **Pyramid / diamond** (patterns 07–10, 12, 17): each row prints spaces + symbols, at most ~2n per row, over n (or 2n) rows → at most `~2n²` prints.

`n²`, `n(n+1)/2 = n²/2 + n/2`, and `2n²` all simplify to the same thing: drop constants, drop smaller terms → **O(n²)** for every single pattern.

In [ ]:
def pattern01(self,n):
    for i in range(n):
        for j in range(n):
            print("*", end="")
        print()

def pattern02(self,n):
    for i in range(n):
        for j in range(i+1):
            print("*", end="")
        print()

### Counting `pattern01` (square) and `pattern02` (triangle)

**pattern01:** the inner loop runs n times for *every* one of the n outer iterations. Prints = `n × n = n²` → **O(n²)**. This is the textbook nested loop.

**pattern02:** the inner loop runs `i+1` times, so the rows print `1, 2, 3, ..., n` stars. Total:

```
1 + 2 + 3 + ... + n = n(n+1)/2
```

For n = 5 that is 15 stars — check it against the actual output. `n(n+1)/2 = n²/2 + n/2` → drop the ½ and the n/2 → **O(n²)**.

Key lesson: "the inner loop is shorter" does **not** save the Big-O. Half a wedding's handshakes is still a wedding's worth of handshakes.

Patterns 03, 04, 05, 06, 11, 13, 14, 15, 16 and 18 are all this same triangle (sometimes flipped, sometimes printing numbers or letters instead of stars) → all **O(n²)**.

In [ ]:
def pattern07(self,n):
    for i in range(n):

        #spaces
        for j in range(n-i-1):
            print(" ", end = "")
        #stars
        for j in range(0,2*i+1):
            print("*", end = "")
        print()

### Counting `pattern07` (pyramid) — spaces count too

Row i prints `(n-i-1)` spaces + `(2i+1)` stars = `n + i` characters. Sum over all rows:

```
sum of (n + i) for i = 0..n-1  =  n·n + (0+1+...+(n-1))  =  n² + n(n-1)/2  ≈  1.5 n²
```

Drop the 1.5 → **O(n²)**. A printed space is just as much work as a printed star.

**pattern09 (diamond)** = a pyramid loop followed by an inverted pyramid loop. Sequential, so they add: O(n²) + O(n²) = O(2n²) = **O(n²)**.

**pattern10** runs its outer loop `2n+1` times with up to n prints per row → ≤ `(2n+1)·n` → still **O(n²)**. More rows changes the constant, not the shape. Patterns 08, 12 and 17 follow the same reasoning.

### Space complexity of all 18 patterns — O(1)

To state space complexity, ask: **how many things is my function *remembering* at once, and does that grow with n?**

Look through every pattern function: the only variables are loop counters (`i`, `j`) and sometimes one helper (`num` in pattern11/13, `char`/`chars` in 14–18). That is a fixed handful of variables whether n = 5 or n = 5,000.

- No list of size n is built.
- No string of the whole pattern is stored — characters go straight to the screen and are forgotten.
- Printed output does **not** count as space complexity; only memory the program holds counts.

So every pattern uses **O(1) auxiliary space** (auxiliary = extra memory beyond the input).

Note: if I had built each row as a string first (e.g. `"*" * i`) that row string is O(n) space. Still fine, but worth saying in an interview.

## Audit result

| Group | Time | Space |
|---|---|---|
| `Sum_of_first_N` | O(n) | O(1) |
| `reverse_a_number`, `count_digits`, `palindrome_number`, `armstrong_number` | O(log₁₀ n) | O(1) |
| Patterns 01–18 | O(n²) | O(1) |

The habit to keep: after solving *any* problem, ask two questions — **how many times does each loop run?** and **what am I storing that grows with n?** That's the whole audit.